In [27]:
from dataclasses import dataclass
import jax
import optax
from jax import random, numpy as jnp
from jax import vmap, jacfwd, jit
from flax import linen as nn
from evojax.util import get_params_format_fn
import os
import time
import numpy as np
from scipy import io

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
#jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_matmul_precision", "highest")

In [28]:
class PINN(nn.Module):
    n_nodes: int
    def setup(self):
        self.layers = [nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       jnp.sin,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu]
        self.last_layer = nn.Dense(1, kernel_init = jax.nn.initializers.he_uniform(), use_bias=False)

    @nn.compact
    def __call__(self, inputs):
        # split the two variables, probably just by slicing
        t, x = inputs[:,0:1], inputs[:,1:2]
        def get_u(t, x):
            f = jnp.hstack([t, x])
            for i, lyr in enumerate(self.layers):
                f = lyr(f)
                if (i == 0):
                    f = 2 *jnp.pi *f
            u = self.last_layer(f)
            return u

        u = get_u(t, x)

        # obtain u_t, u_xx
        def get_u_der(get_u, t, x):
            u_t, u_x = jacfwd(get_u)(t, x), jacfwd(get_u, argnums=1)(t, x)
            u_xx = jacfwd(jacfwd(get_u, argnums=1), argnums=1)(t, x)
            return u_t, u_x, u_xx

        f_der_vmap = vmap(get_u_der, in_axes=(None, 0, 0))
        u_t, u_x, u_xx = f_der_vmap(get_u, t, x)
        u_t, u_x, u_xx = u_t[:,:,0], u_x[:,:,0], u_xx[:,:,0,0]

        # obtain BC/IC indices
        ic, bc = (t == t_l), (x == x_l) # (x==x_l2)  # lower x
        pbc = get_u(t, jnp.ones_like(t)*x_u) - u
        nbc = get_u(t, jnp.ones_like(t)*x_u2) - get_u(t, jnp.ones_like(t)*x_l2)

        # PDE: u_t - v2* u_xx + 5*(u**3) - 5*u = 0
        pde = (u_t - v2* u_xx + 5*(u**3) - 5*u)  

        outputs = jnp.hstack([u, u_xx, pde, pbc, nbc, ic, bc])
        return outputs
    

class DNN(nn.Module):
    """DNNs"""
    n_nodes: int
    def setup(self):
        self.layers = [nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       jnp.sin,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu]
        self.last_layer = nn.Dense(1, kernel_init = jax.nn.initializers.he_uniform(), use_bias=False)

    @nn.compact
    def __call__(self, inputs):
        # split the two variables, probably just by slicing
        t, x = inputs[:,0:1], inputs[:,1:2]
        def get_u(t, x):
            f = jnp.hstack([t, x])
            for i, lyr in enumerate(self.layers):
                f = lyr(f)
                if (i == 0):
                    f = 2 *jnp.pi *f
            u = self.last_layer(f)
            return u

        u = get_u(t, x)
        def get_u0_der(get_u, t, x):
            u_xx = jacfwd(jacfwd(get_u, argnums=1), argnums=1)(t, x)
            return  u_xx
        
        f_der_vmap = vmap(get_u0_der, in_axes=(None, 0, 0))
        u_xx = f_der_vmap(get_u, t, x)[:,:,0,0]

        outputs = jnp.hstack([u, u_xx])
        return outputs

In [29]:
def main(ER, ER_xx, weight_ic, weight_bc, max_lr, exponent, max_iter, seed, gpu):
    global v2, t_l, t_u, x_l, x_u, x_l2, x_u2
    v2 = 0.0001
    mat_data = io.loadmat('E:/SCALE-PINN/ac/allen_cahn.mat')
    sim_t = np.repeat(mat_data['t'], 512).reshape(-1,1)
    sim_x = np.tile(mat_data['x'], 201).reshape(-1,1)
    sim_u = mat_data['usol'].reshape(-1,1)
    tt, xx = mat_data['t'], mat_data['x']

    t_l, t_u, x_l, x_u = np.min(sim_t), np.max(sim_t), np.min(sim_x), np.max(sim_x)
    dt, dx = tt[0,1] - tt[0,0], xx[0,1] - xx[0,0]
    x_l2, x_u2 = x_l + dx, x_u - dx
    
    data_X, data_Y = np.hstack([sim_t, sim_x]), sim_u

    bic = (data_X[:,0] == 0) | (data_X[:,1] == x_l)
    data_X_BIC, data_Y_BIC = data_X[bic], data_Y[bic]

    data_X, data_Y, data_X_BIC, data_Y_BIC = jnp.array(data_X), jnp.array(data_Y), jnp.array(data_X_BIC), jnp.array(data_Y_BIC)

    seed = seed
    key, rng = random.split(random.PRNGKey(seed))
    a = random.normal(key, [1,2])

    n_nodes = 128
    model, model_0 = PINN(n_nodes), DNN(n_nodes)
    params = model.init(key, a)
    num_params, format_params_fn = get_params_format_fn(params)
    params = jax.flatten_util.ravel_pytree(params)[0]
    params_0 = params

    BS_ALL = 1024
    BS_BIC = 64

    n_all, n_bic = len(data_X), len(data_X_BIC)

    @jit
    def minibatch(key):
        key1, key2 = key
        batch_all = random.choice(key1, n_all, (BS_ALL - BS_BIC,))
        batch_bic = random.choice(key2, n_bic, (BS_BIC,))
        batch_X = jnp.vstack([data_X[batch_all], data_X_BIC[batch_bic]])
        batch_Y = jnp.vstack([data_Y[batch_all], data_Y_BIC[batch_bic]])
        return (batch_X, batch_Y)
    
    def eval_loss(params, params_0, inputs, labels):
        pred = model.apply(format_params_fn(params), inputs)
        u, u_xx, pde, pbc, nbc, ic, bc = jnp.split(pred, 7, axis=1)
        pred_0 = model_0.apply(format_params_fn(params_0), inputs)
        u_0, u0_xx = jnp.split(pred_0, 2, axis=1)
        if (ER > 0):
            pde = pde + (u - u_0) / ER - v2*(u_xx - u0_xx) / ER_xx
        pde_loss = jnp.mean(jnp.square(pde))
        ic_e = (labels - u) * ic
        ic_loss = jnp.sum(jnp.square(ic_e)) / ic.sum()
        pbc_e, nbc_e = pbc * bc, nbc * bc
        bc_loss = jnp.sum(jnp.square(pbc_e)) / bc.sum() + jnp.sum(jnp.square(nbc_e)) / bc.sum()

        mse = jnp.mean(jnp.square(labels - u))
        rl2 = jnp.linalg.norm(labels - u) / jnp.linalg.norm(labels)
        loss = pde_loss + weight_ic* ic_loss + weight_bc* bc_loss
        return loss, (mse, rl2)
    
    loss_grad = jax.jit(jax.value_and_grad(eval_loss, has_aux=True))

    @jit
    def update(params, params_0, opt_state, key):
        batch_X, batch_Y = minibatch(key)
        (loss, (mse, rl2)), grad = loss_grad(params, params_0, batch_X, batch_Y)
        updates, opt_state = optimizer.update(grad, opt_state)
        params_0 = params
        params = optax.apply_updates(params, updates)
        return params, params_0, opt_state, loss, mse, rl2
    
    lr_scheduler = optax.warmup_cosine_decay_schedule(init_value=max_lr, peak_value=max_lr, warmup_steps=0, decay_steps=max_iter, end_value=1e-10, exponent=exponent)
    optimizer = optax.adam(learning_rate=lr_scheduler)
    opt_state = optimizer.init(params)

    runtime = 0
    train_iters = 0

    store_nnws = []
    store_nnws.append([0, format_params_fn(params)])
    
    while (train_iters <= max_iter) and (runtime < 2000):
        start = time.time()
        key1, key2, rng = random.split(rng, 3)
        params, params_0, opt_state, loss, mse, rl2 = update(params, params_0, opt_state, (key1, key2))
        end = time.time()
        runtime += (end - start)
        if (train_iters % 100 == 0):
            print('iter. = %05d, time = %03ds, loss = %.2e | mse = %.2e, rl2 = %.2e'%(train_iters, runtime, loss, mse, rl2))
            store_nnws.append([runtime, format_params_fn(params)])
        train_iters += 1

    inputs, labels = data_X, data_Y
    pred = model_0.apply(format_params_fn(params), inputs)
    u, _, = jnp.split(pred, 2, axis=1)
    mse = jnp.mean(jnp.square(labels - u))
    rl2 = jnp.linalg.norm(labels - u) / jnp.linalg.norm(labels)
    print('[v2=%.1e]: MSE = %.2e RL2 = %.2e'%(v2, mse, rl2))
    return format_params_fn(params), store_nnws

In [30]:
seed = 10
params_5, store_nnws = main(ER=0.45, ER_xx=1.5, weight_ic=100, weight_bc=100, max_lr=2e-3, exponent=1.2, max_iter=5000, seed=seed, gpu=0)

iter. = 00000, time = 001s, loss = 9.15e+01 | mse = 8.39e-01, rl2 = 1.28e+00
iter. = 00100, time = 010s, loss = 1.34e+00 | mse = 3.29e-01, rl2 = 8.19e-01
iter. = 00200, time = 021s, loss = 3.29e-01 | mse = 2.14e-01, rl2 = 6.60e-01
iter. = 00300, time = 033s, loss = 7.31e-02 | mse = 2.23e-01, rl2 = 6.65e-01
iter. = 00400, time = 045s, loss = 2.16e-01 | mse = 2.48e-01, rl2 = 7.21e-01
iter. = 00500, time = 056s, loss = 8.00e-02 | mse = 2.54e-01, rl2 = 7.02e-01
iter. = 00600, time = 067s, loss = 2.33e-02 | mse = 2.39e-01, rl2 = 6.78e-01
iter. = 00700, time = 079s, loss = 2.04e-02 | mse = 2.27e-01, rl2 = 6.55e-01
iter. = 00800, time = 090s, loss = 3.33e-02 | mse = 2.34e-01, rl2 = 6.92e-01
iter. = 00900, time = 102s, loss = 1.62e-02 | mse = 2.24e-01, rl2 = 6.72e-01
iter. = 01000, time = 113s, loss = 1.57e-02 | mse = 2.06e-01, rl2 = 6.45e-01
iter. = 01100, time = 125s, loss = 1.56e-01 | mse = 2.10e-01, rl2 = 6.54e-01
iter. = 01200, time = 139s, loss = 3.92e-02 | mse = 2.07e-01, rl2 = 6.35e-01

In [31]:
# ==============================
# 物理 / 方程参数
# ==============================
v2: float = 0.0001
ic_lambda: float = 3.0
bc_lambda: float = 1.0

# ==============================
# 计算域范围
# ==============================
x_min: float = -1.0
x_max: float = 1.0
t_min: float = 0.0
t_max: float = 10.0

# ==============================
# 网络结构 TINN
# ==============================
t_layers: list = [1, 10, 10, 5]
layers: list = [2, 20, 20, 1]

# ==============================
# 训练超参数
# ==============================
N_coll: int = 15000
N_ic: int = 4000
N_bc: int = 2000

N_val_coll: int = 12000
N_val_ic: int = 3000
N_val_bc: int = 1200

Epoch: int = 10000
mu: float = 10.0
mu_update: int = 2
div_factor: float = 1.3
mul_factor: float = 1.7
min_mu: float = 1e-12

# ==============================
# 测试网格精度
# ==============================
test_width: float = 1.0
test_step: int = 30
final_test_step: int = 101
time_steps: int = 201

# ==============================
# 随机种子
# ==============================
seed: int = 4

In [32]:
import os
os.environ['JAX_ENABLE_X64'] = 'True'
os.environ['JAX_DEFAULT_DTYPE_BITS'] = '64'
import jax
import jax.numpy as jnp
from jax import jit, random, jvp
import optax
from flax import linen as nn
jax.config.update("jax_enable_x64", True)

class TINNSpaceLayer(nn.Module):
    """封装 TINN 的单空间调制层"""
    out_dim: int
    @nn.compact
    def __call__(self, z, coef):
        # coef: (B, 2) —— 对应这一层的两个调制系数
        dense_a = nn.Dense(
            self.out_dim,
            use_bias=False,
            kernel_init=nn.initializers.zeros
        )
        ab = self.param("ab", nn.initializers.zeros, (1, self.out_dim))
        term1 = dense_a(z) * coef[..., 0:1] + ab * coef[..., 0:1]

        dense_b = nn.Dense(
            self.out_dim,
            kernel_init=nn.initializers.xavier_uniform(),
            bias_init=nn.initializers.zeros
        )
        term2 = dense_b(z)
        return nn.tanh(term1 + term2)

class TINN(nn.Module):
    t_layers: list
    layers: list
    @nn.compact
    def __call__(self, xt):
        x = xt[..., :2]
        t = xt[..., 2:3]

        temp_t = t
        for feat in self.t_layers[1:-1]:
            temp_t = nn.Dense(feat, kernel_init=nn.initializers.xavier_uniform())(temp_t)
            temp_t = nn.tanh(temp_t)
        temp_t = nn.Dense(self.t_layers[-1], kernel_init=nn.initializers.xavier_uniform())(temp_t)

        t_alpha = self.param("t_alpha", nn.initializers.ones, (1, self.t_layers[-1]))
        temp_t = (1.0 - t_alpha) * t + t_alpha * temp_t
        
        z = x
        # 构建空间层序列
        space_layers = []
        for i in range(len(self.layers)-2):
            space_layers.append(TINNSpaceLayer(out_dim=self.layers[i+1]))
        
        # 前向传播，每一层取对应的 temp_t 系数
        for i, layer in enumerate(space_layers):
            # 取这一层对应的两个系数: 2i 和 2i+1
            coef = temp_t[..., 2*i : 2*i+2]
            z = layer(z, coef)

        dense_a_out = nn.Dense(
            self.layers[-1],
            use_bias=False,
            kernel_init=nn.initializers.zeros
        )
        dense_b_out = nn.Dense(
            self.layers[-1],
            kernel_init=nn.initializers.xavier_uniform(),
            use_bias=False
        )
        coef_out = temp_t[..., -1:]
        term1 = dense_a_out(z) * coef_out
        term2 = dense_b_out(z)
        y = term1 + term2
        return y
    

In [33]:
model = TINN(t_layers=t_layers, layers=layers)
def exact_u(xt):
    x1 = xt[...,0:1]
    x2 = xt[...,1:2]
    t = xt[...,2:3]
    return (x1 + x2)*jnp.cos(2*t) + x1*x2*jnp.sin(2*t)

def f_source(xt):
    return -4*exact_u(xt) + exact_u(xt)**2

def get_der(params, xt):
    v_x = jnp.zeros_like(xt).at[:, 0].set(1.0)  # x方向切向量
    v_y = jnp.zeros_like(xt).at[:, 1].set(1.0)  # y方向切向量
    v_t = jnp.zeros_like(xt).at[:, 2].set(1.0)  # t方向切向量
    def get_u(xt):
        return model.apply({'params': params}, xt)
    u = get_u(xt)
    
    def get_u_x(xt):
        _, u_x = jvp(get_u, (xt,), (v_x,))
        return u_x
    def get_u_xx(xt):
        u_x, u_xx = jvp(get_u_x, (xt,), (v_x,))
        return u_x, u_xx
    u_x, u_xx = get_u_xx(xt)

    def get_u_y(xt):
        _, u_y = jvp(get_u, (xt,), (v_y,))
        return u_y
    def get_u_yy(xt):
        u_y, u_yy = jvp(get_u_y, (xt,), (v_y,))
        return u_y, u_yy
    u_y, u_yy = get_u_yy(xt)

    def get_u_t(xt):
        _, u_t = jvp(get_u, (xt,), (v_t,))
        return u_t
    def get_u_tt(xt):
        u_t, u_tt = jvp(get_u_t, (xt,), (v_t,))
        return u_t, u_tt
    u_t, u_tt = get_u_tt(xt)

    return u, u_x, u_xx, u_y, u_yy, u_t, u_tt

def pde_loss(params, xt):
    u, u_x, u_xx, u_y, u_yy, u_t, u_tt = get_der(params, xt)
    f = u_tt - u_xx - u_yy + u**2 - f_source(xt)
    return jnp.mean(f**2)

def data_loss(params, xt):
    u = model.apply({'params': params}, xt)
    u_exact = exact_u(xt)
    return jnp.mean((u - u_exact)**2)

def total_loss(params, coll_xt, ic_xt, bc_xt):
        loss_pde = pde_loss(params, coll_xt)
        loss_ic = data_loss(params, ic_xt)
        loss_bc = data_loss(params, bc_xt)
        loss = loss_pde + ic_lambda * loss_ic + bc_lambda * loss_bc
        return loss

def sample_collocation(rng, N):
    k1, k2 = random.split(rng)
    x = random.uniform(k1, (N, 2), minval=x_min, maxval=x_max)
    t = random.uniform(k2, (N, 1), minval=t_min, maxval=t_max)
    xt = jnp.hstack([x, t])
    return xt

def sample_initial(rng, N):
    k1, k2 = random.split(rng, 2)
    x = random.uniform(k1, (N, 1), minval=x_min, maxval=x_max)
    y = random.uniform(k2, (N, 1), minval=x_min, maxval=x_max)
    t = jnp.zeros_like(x)
    xt = jnp.hstack([x, y, t])
    return xt

def sample_boundary(rng, N):
    k1, k2, k3 = random.split(rng, 3)
    t = random.uniform(k1, (N, 1), minval=t_min, maxval=t_max)
    x = random.uniform(k2, (N, 1), minval=x_min, maxval=x_max)
    y = random.uniform(k3, (N, 1), minval=x_min, maxval=x_max)
    x1 = jnp.ones_like(x)
    pts1 = jnp.hstack([-x1, y, t])
    pts2 = jnp.hstack([x1, y, t])
    pts3 = jnp.hstack([x, -x1, t])
    pts4 = jnp.hstack([x, x1, t])
    xt = jnp.vstack([pts1, pts2, pts3, pts4])
    return xt

x = jnp.linspace(x_min, x_max, test_step)
y = jnp.linspace(x_min, x_max, test_step)
t = jnp.linspace(t_min, t_max, test_step)
XX, YY, TT = jnp.meshgrid(x, y, t, indexing='ij')
X_test = jnp.hstack((XX.flatten().reshape(-1,1), YY.flatten().reshape(-1,1), TT.flatten().reshape(-1,1)))
Z_exact = exact_u(X_test)

def L2Error(params):
    Z_pred = model.apply({'params': params}, X_test)
    Z_error = jnp.abs(Z_pred - Z_exact)
    l2_error = jnp.linalg.norm(Z_error)
    rel_l2_error = l2_error / jnp.linalg.norm(Z_exact)
    return rel_l2_error

In [34]:
key = random.PRNGKey(seed)
xt_dummy = jnp.zeros((1,3))
params = model.init(key, xt_dummy)['params']
total_number = sum(jax.tree_util.tree_leaves(jax.tree_util.tree_map(jnp.size, params)))
print('Total number of parameters: %d'%total_number)

key, kc, ki, kb = random.split(key, 4)
batch_coll_xt = sample_collocation(kc, N_coll)
batch_ic_xt = sample_initial(ki, N_ic)
batch_bc_xt = sample_boundary(kb, N_bc)

def train(steps, params, key):
    opt = optax.adam(learning_rate=1e-3)
    opt_state = opt.init(params)
    
    @jit
    def step(params, opt_state, key):
        key_c, key_i, key_b = random.split(key, 3)
        coll_xt = sample_collocation(key_c, N_coll)
        ic_xt = sample_initial(key_i, N_ic)
        bc_xt = sample_boundary(key_b, N_bc)

        loss, grads = jax.value_and_grad(total_loss)(params, coll_xt, ic_xt, bc_xt)
        updates, opt_state = opt.update(grads, opt_state)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss

    for i in range(steps):
        key, subkey = random.split(key)
        params, opt_state, loss = step(params, opt_state, subkey)
        if (i % 100 == 0):
            rel_l2_error = L2Error(params)
            print(f"Step {i}, Relative L2 Error: {rel_l2_error:.6f}, Loss: {loss:.2e}")
    return params
trained_params = train(Epoch, params, key)
test_rel_l2_error = L2Error(trained_params)
print(f"Final Relative L2 Error: {test_rel_l2_error:.6f}")

Total number of parameters: 1190
Step 0, Relative L2 Error: 1.222098, Loss: 1.18e+01
Step 100, Relative L2 Error: 1.122020, Loss: 7.78e+00
Step 200, Relative L2 Error: 1.002430, Loss: 6.56e+00
Step 300, Relative L2 Error: 1.073740, Loss: 6.01e+00
Step 400, Relative L2 Error: 1.056323, Loss: 5.29e+00
Step 500, Relative L2 Error: 0.996535, Loss: 5.29e+00
Step 600, Relative L2 Error: 0.913812, Loss: 4.74e+00
Step 700, Relative L2 Error: 0.870697, Loss: 4.48e+00
Step 800, Relative L2 Error: 0.867624, Loss: 4.00e+00
Step 900, Relative L2 Error: 0.916671, Loss: 2.98e+00
Step 1000, Relative L2 Error: 0.827068, Loss: 2.04e+00
Step 1100, Relative L2 Error: 0.696731, Loss: 1.43e+00
Step 1200, Relative L2 Error: 0.666563, Loss: 1.33e+00
Step 1300, Relative L2 Error: 0.636506, Loss: 1.16e+00
Step 1400, Relative L2 Error: 0.613993, Loss: 1.15e+00
Step 1500, Relative L2 Error: 0.578242, Loss: 1.06e+00
Step 1600, Relative L2 Error: 0.561436, Loss: 1.03e+00
Step 1700, Relative L2 Error: 0.547961, Loss

KeyboardInterrupt: 

In [ ]:
import os
os.environ["JAX_ENABLE_X64"] = "True"
os.environ["JAX_DEFAULT_DTYPE_BITS"] = "64"

import time
import jax
import jax.numpy as jnp
from jax import jit, random, jvp
import optax
from flax import linen as nn
from flax.training import train_state
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
jax.config.update('jax_enable_x64', True)
ic_lambda = 3.
bc_lambda = 1.

class TINN(nn.Module):
    t_layers: list
    layers: list
    @nn.compact
    def __call__(self, xt):
        xt = jnp.asarray(xt)
        is_scalar = (xt.ndim == 1)
        if is_scalar:
            xt = xt.reshape(1,3)
        x = xt[:, 0:2]
        t = xt[:, 2:3]
        temp_t = t
        for i in range(len(self.t_layers)-2):
            tW = self.param(f"tW{i}", nn.initializers.xavier_uniform(), (self.t_layers[i], self.t_layers[i+1]))
            tb = self.param(f"tb{i}", nn.initializers.zeros, (self.t_layers[i+1],))
            temp_t = jnp.tanh(jnp.matmul(temp_t, tW) + tb)

        tWout = self.param("tWout", nn.initializers.xavier_uniform(), (self.t_layers[-2], self.t_layers[-1]))
        temp_t = jnp.matmul(temp_t, tWout)

        t_alpha = self.param("t_alpha", nn.initializers.ones, (1, self.t_layers[-1]))
        temp_t = ((1.0 - t_alpha) * t) + (t_alpha * temp_t)
        z = x
        for i in range(len(self.layers)-2):
            in_dim = self.layers[i]
            out_dim = self.layers[i+1]

            aW = self.param(f"aW{i}", nn.initializers.zeros, (in_dim, out_dim))
            ab = self.param(f"ab{i}", nn.initializers.xavier_uniform(), (1,out_dim))
            bW = self.param(f"bW{i}", nn.initializers.xavier_uniform(), (in_dim, out_dim))
            bb = self.param(f"bb{i}", nn.initializers.zeros, (out_dim,))
            coef = temp_t[..., 2*i:2*i+1]
            term1 = jnp.matmul(z, aW) * coef
            term2 = jnp.matmul(z, bW)
            coef = temp_t[..., 2*i+1:2*i+2]
            b_eff = ab * coef + bb
            z = jnp.tanh(term1 + term2 + b_eff)
        # output layer
        aW_out = self.param("aWout", nn.initializers.zeros, (self.layers[-2], self.layers[-1]))
        bW_out = self.param("bWout", nn.initializers.xavier_uniform(), (self.layers[-2], self.layers[-1]))
        coef_out = temp_t[..., -1:]
        term1 = jnp.matmul(z, aW_out) * coef_out
        term2 = jnp.matmul(z, bW_out)
        y = term1 + term2
        y = jnp.squeeze(y, axis=-1)
        if is_scalar:
            return y[0]
        return y

def exact_u(xt):
    return (xt[0] + xt[1])*jnp.cos(2*xt[2]) + xt[0]*xt[1]*jnp.sin(2*xt[2])

def f_source(xt):
    return -4*exact_u(xt) + exact_u(xt)**2

def make_jvp_kernels(model):
    def u_batch(params, xt_batch):
        return model.apply({'params': params}, xt_batch)

    @jax.jit
    def u_xx_jvp(params, xt_batch):
        N = xt_batch.shape[0]
        v_x = jnp.tile(jnp.array([1.0, 0.0, 0.0]), (N,1))
        def first_tangent(xs):
            _, tangent = jvp(lambda x_in: u_batch(params, x_in), (xs,), (v_x,))
            return tangent
        _, u_xx = jvp(first_tangent, (xt_batch,), (v_x,))
        return u_xx
    
    @jax.jit
    def u_yy_jvp(params, xt_batch):
        N = xt_batch.shape[0]
        v_x = jnp.tile(jnp.array([0.0, 1.0, 0.0]), (N,1))
        def first_tangent(xs):
            _, tangent = jvp(lambda x_in: u_batch(params, x_in), (xs,), (v_x,))
            return tangent
        _, u_yy = jvp(first_tangent, (xt_batch,), (v_x,))
        return u_yy
    
    @jax.jit
    def u_tt_jvp(params, xt_batch):
        N = xt_batch.shape[0]
        v_t = jnp.tile(jnp.array([0.0, 0.0, 1.0]), (N,1))
        def first_tangent(xs):
            _, tangent = jvp(lambda x_in: u_batch(params, x_in), (xs,), (v_t,))
            return tangent
        _, u_tt = jvp(first_tangent, (xt_batch,), (v_t,))
        return u_tt
    
    @jax.jit
    def grads_jvp(params, xt_batch):
        N = xt_batch.shape[0]
        v_t = jnp.tile(jnp.array([0.0, 0.0, 1.0]), (N,1))
        _, u_t = jvp(lambda xs: u_batch(params, xs), (xt_batch,), (v_t,))
        return u_t
    u_batch_jit = jax.jit(u_batch)
    return u_batch_jit, u_xx_jvp, u_yy_jvp, u_tt_jvp, grads_jvp

def build_loss_and_steps(model):
    u_batch, u_xx_jvp, u_yy_jvp, u_tt_jvp, grad_jvp = make_jvp_kernels(model)
    @jax.jit
    def pde_residual_single(params, xt):
        u = u_batch(params, xt[None, :])[0]
        u_xx = u_xx_jvp(params, xt[None, :])
        u_yy = u_yy_jvp(params, xt[None, :])
        u_tt = u_tt_jvp(params, xt[None, :])
        return u_tt - u_xx - u_yy + u**2
    pde_jac_single = jax.jacrev(pde_residual_single, argnums=0)
    diff_jac_single = jax.jacrev(lambda p, xt: (grad_jvp(p, xt[None,:])[0] - 2*xt[0]*xt[1]), argnums=0)
    ic_jac_single  = jax.jacrev(lambda p, xt: ic_lambda*(u_batch(p, xt[None,:])[0] - exact_u(xt)), argnums=0)
    bc_jac_single = jax.jacrev(lambda p, xt: bc_lambda*(u_batch(p, xt[None, :])[0] - exact_u(xt)), argnums = 0 )
    
    def vmap_jac(jac_single, xt_batch, params, chunk=15000):
        N = xt_batch.shape[0]
        outs = []
        for i in range(0, N, chunk):
            xt_sub = xt_batch[i:i+chunk]
            jac_sub = jax.vmap(lambda xt: jac_single(params, xt))(xt_sub)
            outs.append(jac_sub)
        return jax.tree_util.tree_map(lambda *xs: jnp.concatenate(xs, axis=0), *outs)
    
    @jax.jit
    def loss_fn(params, batch_coll_xt, batch_ic_xt, batch_bc_xt, f1):
        res_pde = jax.vmap(lambda xt: pde_residual_single(params, xt))(batch_coll_xt).reshape(len(batch_coll_xt))
        res_pde = res_pde - f1
        res_ic = jax.vmap(
            lambda xt: ic_lambda*(u_batch(params, xt[None,:])[0] - exact_u(xt)))(batch_ic_xt)
        res_bc = jax.vmap(
            lambda xt: bc_lambda*(u_batch(params, xt[None,:])[0] - exact_u(xt)))(batch_bc_xt)
        res_diff_ic = jax.vmap(
            lambda xt: (grad_jvp(params, xt[None,:])[0] - 2*xt[0]*xt[1]))(batch_ic_xt)
        lpde = jnp.mean(res_pde**2)
        lic = jnp.mean(res_ic**2)
        ldiffic = jnp.mean(res_diff_ic**2)
        lbc = jnp.mean(res_bc**2)
        total = lpde+lic + lbc + ldiffic
        return total, (lpde, lic, lbc, ldiffic)
    
    @jax.jit
    def train_step(params, batch_coll_xt, batch_ic_xt, batch_bc_xt, f1):
        res_pde = jax.vmap(lambda xt: pde_residual_single(params, xt))(batch_coll_xt).reshape(len(batch_coll_xt), -1)
        res_pde = res_pde - f1
        res_ic = jax.vmap(
            lambda xt: ic_lambda*(u_batch(params, xt[None,:])[0] - exact_u(xt)))(batch_ic_xt).reshape(len(batch_ic_xt),-1)
        res_bc = jax.vmap(
            lambda xt: bc_lambda*(u_batch(params, xt[None,:])[0] - exact_u(xt)))(batch_bc_xt).reshape(len(batch_bc_xt), -1)
        res_diff_ic = jax.vmap(
            lambda xt: (grad_jvp(params, xt[None,:])[0] - 2*xt[0]*xt[1]))(batch_ic_xt).reshape(len(batch_ic_xt), -1)

        pde_theta = vmap_jac(pde_jac_single, batch_coll_xt, params)
        ic_theta  = vmap_jac(ic_jac_single,  batch_ic_xt, params)
        diff_ic_theta  = vmap_jac(diff_jac_single,  batch_ic_xt, params)
        bc_theta = vmap_jac(bc_jac_single, batch_bc_xt, params)
        return res_pde, pde_theta, res_ic, ic_theta, res_bc, bc_theta, res_diff_ic, diff_ic_theta
    return loss_fn, train_step

def sample_collocation(rng, N):
    k1, k2 = random.split(rng)
    x = random.uniform(k1, (N,2), minval=-1.0, maxval=1.0)
    t = random.uniform(k2, (N,1), minval=0.0, maxval=10.0)
    return jnp.hstack([x,t])

def sample_ic(rng, N):
    k1, k2 = random.split(rng)
    x = random.uniform(k1, (N,1), minval=-1.0, maxval=1.0)
    y = random.uniform(k2, (N,1), minval=-1.0, maxval=1.0)
    t = jnp.zeros_like(x)
    return jnp.hstack([x, y,t])

def sample_bc(rng, N):
    k1, k2, k3 = random.split(rng, 3)
    t = random.uniform(k1, (N,1), minval=0.0, maxval=10.0)
    x = random.uniform(k2, (N,1), minval = -1.0, maxval = 1.0)
    y = random.uniform(k3, (N,1), minval = -1.0, maxval = 1.0)
    x1 =  jnp.ones_like(t)
    pts0 = jnp.hstack([-x1, y, t])
    pts1 = jnp.hstack([x1, y, t])
    pts2 = jnp.hstack([x, -x1, t])
    pts3 = jnp.hstack([x, x1, t])
    return jnp.vstack([pts0, pts1, pts2, pts3])

# error for training check (fewer points)
width = 1
step = 30
x = jnp.linspace(-width, width, step)
y = jnp.linspace(-width, width,step)
t = jnp.linspace(0,10,21)
XX, YY, TT = jnp.meshgrid(x, y, t, indexing="ij")
total_test = len(XX.flatten())
X_test = jnp.hstack((XX.flatten().reshape(total_test,1), YY.flatten().reshape(total_test,1), TT.flatten().reshape(total_test,1)))
Z_exact = jax.vmap(exact_u,(0))(X_test).reshape(XX.shape)
def L2Error():
    Z_pred = jax.vmap(model.apply, (None, 0))({'params': params}, X_test).reshape(XX.shape)
    Z_error = abs(Z_pred - Z_exact)
    # l2 error
    l2_error = np.sqrt(sum(Z_error.flatten()**2)/(Z_error.size))
    rel_l2_error = l2_error / np.sqrt(np.sum(Z_exact.flatten()**2)/Z_error.size)
    return rel_l2_error

key = random.PRNGKey(4)
model = TINN([1,10,10,5], [2,20,20,1])
xt_dummy = jnp.zeros((1,3))
params = model.init(key, xt_dummy)['params']
total_number = 0
for p in params:
    total_number +=params[p].size
print("total parameter:", total_number)

loss_fn, train_step = build_loss_and_steps(model)
_dummy_coll = sample_collocation(key, 8)
_dummy_f = jax.vmap(f_source, 0)(_dummy_coll)
_dummy_ic = sample_ic(key, 4)
_dummy_bc = sample_bc(key, 4)
_ = loss_fn(params, _dummy_coll, _dummy_ic, _dummy_bc, _dummy_f)
res_pde, pde_theta, res_ic, ic_theta, res_bc, bc_theta, res_diff, diff_ic_theta = train_step(params, _dummy_coll, _dummy_ic, _dummy_bc, _dummy_f)

N_coll = 15000
N_ic   = 4000
N_bc = 2000
N_val_coll = 12000
N_val_ic = 3000
N_val_bc = 1200
key, kc, ki, kb = random.split(key, 4)
batch_coll_xt = sample_collocation(kc, N_coll)
f1_batch = jax.vmap(f_source,0)(batch_coll_xt).reshape(N_coll, -1)
batch_ic_xt   = sample_ic(ki, N_ic)
batch_bc_xt = sample_bc(kb, N_bc)
key, vk1, vk2, vk3 = random.split(key, 4)
val_coll_xt = sample_collocation(vk1, N_val_coll)
val_f1 = jax.vmap(f_source,0)(val_coll_xt)
val_ic_xt   = sample_ic(vk2, N_val_ic)
val_bc_xt   = sample_bc(vk3, N_val_bc)
# ic_total = jax.vmap(exact_u,(0))(batch_ic_xt)
# ic_average = jnp.mean(jnp.sort(abs(ic_total))[:2400])
# print(1/(ic_average))

def LM_reshape(grad_params):
    N_num = grad_params['aW0'].shape[0]
    temp = grad_params['aW0'].reshape(N_num, jnp.size(grad_params['aW0'])//N_num)
    for ii, p in enumerate(grad_params):
        if ii>0:
            temp = jnp.hstack((temp, grad_params[p].reshape(N_num, jnp.size(grad_params[p])//N_num )))
    return temp
#-------------------------------
Epoch = 10000
mu = 10**1
itera_ = 0
mu_update = 2
div_factor = 1.3
mul_factor = 1.7
loss_sum_old = 10**5
min_mu = 10**-12
loss_LM = []
l2_error_list = []
#-------------------------------
start = time.time()
None_valid = 0
for step in range(Epoch):
    res_pde, pde_theta, res_ic, ic_theta, res_bc, bc_theta, res_diff, diff_ic_theta = train_step(params, batch_coll_xt, batch_ic_xt, batch_bc_xt, f1_batch)
    re_pde_theta = LM_reshape(pde_theta)
    re_ic_theta = LM_reshape(ic_theta)
    re_bc_theta = LM_reshape(bc_theta)
    re_diff_theta = LM_reshape(diff_ic_theta)
    val_tot, (val_pde, val_ic, val_bc, val_diff) = loss_fn(params, val_coll_xt, val_ic_xt, val_bc_xt, val_f1)
    # J_mat
    J_mat = jax.lax.concatenate((re_pde_theta/(N_coll**0.5), re_ic_theta/N_ic**0.5, re_bc_theta/(N_bc)**0.5, re_diff_theta/N_ic**0.5), 0)
    #L_vec
    L_vec = jax.lax.concatenate((res_pde/N_coll**0.5, res_ic/N_ic**0.5, res_bc/(N_bc)**0.5, res_diff/N_ic**0.5 ), 0)
    loss = jnp.mean(res_ic**2) + jnp.mean(res_pde**2) + jnp.mean(res_bc**2) + jnp.mean(res_diff**2)
    loss_LM.append(loss.item())
    I = jnp.eye((J_mat.shape[1]))
    J_product = J_mat.T@J_mat
    rhs = -J_mat.T@L_vec
    dp = jnp.linalg.solve((J_product+mu*I) , rhs)
    cnt=0
    for p in pde_theta:
        num = jnp.size(params[p])
        params[p] = params[p] + dp[cnt:cnt+num].reshape(params[p].shape)
        cnt+=num
    itera_ += 1
    if step % mu_update == 0:
        if loss < loss_sum_old:
            mu = max(mu/div_factor, min_mu)
        else:
            mu = min(mul_factor*mu, 10**(8))
        loss_sum_old = loss
    if loss.item()/mu > 10**5:
        mu = loss.item()/10
    if step%50 == 0:
        lpde = jnp.mean(res_pde**2)
        lic = jnp.mean(res_ic**2)
        lbc = jnp.mean(res_bc**2)
        ldiff = jnp.mean(res_diff**2)
        elapsed = time.time()-start
        temp_error = L2Error().item()
        l2_error_list.append(temp_error)
        print(f"Epoch {itera_:4d} | L2Error={temp_error:.3e} | loss={loss:.3e} | val={val_tot:.3e} | mu={mu:.3e} | pde={lpde:.3e} | ic={lic:.3e} | bc={lbc:.3e} | diff={ldiff:.3e} | time={elapsed:.1f}s")
    if (val_tot/loss>5):
        print('Validation')
        key, kc, ki, kb = random.split(key, 4)
        batch_coll_xt = sample_collocation(kc, N_coll)
        f1_batch = jax.vmap(f_source,0)(batch_coll_xt).reshape(N_coll, -1)
        batch_ic_xt   = sample_ic(ki, N_ic)
        batch_bc_xt = sample_bc(kb, N_bc)
        None_valid = 0
print('the final Loss: %.5e'% (loss))
print('the final valid Loss: %.5e'% (val_tot))
final_time = time.time()
print('total time:', final_time - start)
print("final min mu:", min_mu)

#Error
width = 1
step = 101
x = jnp.linspace(-width, width, step)
y = jnp.linspace(-width, width,step)
XX, YY = jnp.meshgrid(x, y, indexing="ij")
total_test = len(XX.flatten())
X_test = jnp.hstack((XX.flatten().reshape(total_test,1), YY.flatten().reshape(total_test,1)))
Z_exact = np.zeros([101,101,201])
Z_error = np.zeros([101,101,201])
Z_pred = np.zeros([101,101,201])
for ii in range(201):
    X_time_test = jnp.hstack((X_test, jnp.tile(jnp.array([ii*0.05]), [total_test, 1])))
    Z_exact[:,:,ii]+= jax.vmap(exact_u,(0))(X_time_test).reshape(XX.shape)
    Z_pred[:,:,ii] += jax.vmap(model.apply, (None, 0))({'params': params}, X_time_test).reshape(XX.shape)
    Z_error[:,:,ii]+= abs(Z_pred[:,:,ii]-Z_exact[:,:,ii])
# l2 error
l2_error = np.sqrt(sum(Z_error.flatten()**2)/(201*101**2))
l_int_error = max(abs(Z_error.flatten()))
rel_l2_error = l2_error / np.sqrt(sum(Z_exact.flatten()**2)/(201*101**2))
rel_l_inf_error = l_int_error / max(abs(Z_exact.flatten()))

print('L2-Error:%.5e'%l2_error)
print('Linf-Error:%.5e'%l_int_error)
print('rel-L2-Error: %.5e'% rel_l2_error)
print('rel-Linf-Error: %.5e'% rel_l_inf_error)

total parameter: 1185
Epoch    1 | L2Error=1.081e+00 | loss=1.223e+01 | val=1.213e+01 | mu=7.692e+00 | pde=6.724e+00 | ic=4.114e+00 | bc=9.382e-01 | diff=4.571e-01 | time=8.7s
Epoch   51 | L2Error=1.075e-01 | loss=4.177e-02 | val=4.184e-02 | mu=2.600e-01 | pde=3.563e-02 | ic=6.871e-04 | bc=4.165e-03 | diff=1.287e-03 | time=126.5s


KeyboardInterrupt: 